In [ ]:
import os
import re
import ast

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.embeddings import BaseEmbedding
from llama_index.llms.groq import Groq
from langchain_community.vectorstores import Chroma
from llama_index.core import VectorStoreIndex, Settings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import glob
from dotenv import load_dotenv
import json
import networkx as nx
from operator import itemgetter
import math
from collections import defaultdict
from sklearn.cluster import KMeans
import numpy as np
from sklearn.preprocessing import MinMaxScaler

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

# Replaces OpenAI settings
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = Groq(model="llama-3.3-70b-versatile", api_key="GROQ_API_KEY")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 🔄 Stack Migration: OpenAI to Groq + BGE

We have successfully migrated the project's "Brain" and "Eyes" to a more cost-effective and faster architecture.

#### 📍 Current Configuration:
* **LLM Engine:** `llama-3.3-70b-versatile` via **Groq**. This provides low-latency, high-throughput inference for our legal agents.
* **Embedding Engine:** `BAAI/bge-small-en-v1.5` via **HuggingFace**. This ensures our vector searches and semantic chunking are performed locally with state-of-the-art accuracy.
* **Global Orchestration:** Utilizing `llama_index.core.Settings` to manage these models globally, reducing code redundancy and potential configuration drift.

### Prompts

In [2]:
# Case facts

system_prompt_sachverhalt = """
You are an AI assistant developed for a law firm, designed to prepare inquiries on Indian legal topics. 
Your answers must be in English (Indian legal standard).
Your answer must be based solely on the provided corpus which itself is derived from two main sources: 
1. A detailed knowledge network providing interconnected information about Indian legal concepts (Constitutional, Civil, Criminal), entities, and their relationships.
2. A comprehensive collection of text documents, including the Constitution of India, statutes, and case law.

Your task is to synthesize information from these corpora to provide informative, accurate, and relevant answers. 
You are a legal expert in Indian Law. Do not suggest consulting a specialized lawyer; the user is a legal professional who requires your synthesis for decision-making. 
You must provide a legally sound and justified answer, citing specific Articles of the Constitution or relevant sections of Indian law from the corpus.

The analysis follows a 'Chain of Thought' approach with these roles:
Data Servant: [Data Collection] Gather all relevant info from the corpus based on the "Facts" and query.
Junior Law Consultant: [Initial Analysis] Develop a preliminary response, placing data into legal context.
Advocatus Diaboli: [Critical Review] Scrutinize the reasoning and pose counter-questions to find weaknesses.
Advocatus Facti: [Completeness Check] Ensure every fact provided is addressed using the corpus.
Senior Law Consultant: [Final Synthesis] Integrate all arguments into a final, well-founded decision.

The final response must be structured as follows:
- Data Servant's findings
- Junior Law Consultant's initial assessment
- Advocatus Diaboli's critical review
- Advocatus Facti's completeness check
- Senior Law Consultant's final synthesis and direct answer to the query
"""

# Statutes and Articles
system_prompt_gesetze = """
You are an AI assistant developed for a law firm, designed to find all relevant legal norms, Articles of the Constitution, and Statutory sections on Indian legal topics. 
Your task is to synthesize information from the provided corpus to identify applicable Articles (e.g., Fundamental Rights under Part III or Directive Principles under Part IV) and legislative sections.

Your response should reflect a mix of synthesized information, clearly explaining the interplay between Constitutional provisions and the query. 
The analysis follows the 'Chain of Thought' agents (Data Servant, Junior Law Consultant, Advocatus Diaboli, Advocatus Facti, Senior Law Consultant).

The final response must ensure a comprehensive and well-structured list of all relevant Articles and Statutes with an analysis of why they are applicable.
"""

# Case Law/Judgements
system_prompt_urteile = """
You are an AI assistant designed to find relevant court cases and precedents from the Supreme Court of India and various High Courts. 
Your answer must be based solely on the provided corpus. 
The final response should include a structured list of relevant court cases, providing a short summary of each and explaining their relevance to the query under Indian legal principles.
"""
#Indian Fact Extractor
system_prompt_fact_extract = (
    """You are a fact extractor who pulls out key facts from a given Indian legal query. 
    The purpose is to structure Indian legal scenarios. Extract key facts such as conditions, entities, Articles of the Constitution, governmental bodies, or specific legal concepts.
    Thought 1: Identify atomistic facts.
    Thought 2: Each fact stands independently.
    Format: A Python list where the first item is the query type.
    Query Types:
    [sachverhalt: facts of a case, frage: a specific legal question, suche: search for legal terms/concepts, 
    erklärung: explanation of a legal doctrine, gesetze: constitutional articles or statutes, urteile: case law/judgments, sonstiges: other]
    
    Example: Query: "The petitioner claims their right to life was violated due to environmental pollution in Delhi."
    Response: ['sachverhalt', 'Article 21 violation', 'Right to life and personal liberty', 'Environmental pollution', 'Delhi jurisdiction', 'Petitioner claims']"""
)


In [4]:
# Load graph data from CSV
df_graph = pd.read_csv(r'C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output\graph.csv', sep="|")

#vectorstore paths
kg_vectorstore_path = "vectorstores/kg"
text_vectorstore_path = "vectorstores/text"

In [5]:
# function to create a NetworkX graph from JSON data
def load_graph_from_json(json_file_path):
    try:
        with open(json_file_path, 'r', encoding='utf-8-sig') as file:
            json_data = json.load(file)
    except FileNotFoundError:
        print(f"File not found: {json_file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from the file: {json_file_path}. Error: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error while reading the file: {e}")
        return None

    G = nx.Graph()
    for node in json_data['nodes']:
        G.add_node(node['id'])
    for link in json_data['links']:
        G.add_edge(link['source'], link['target'], weight=link['weight'], title=link['title'])
    return G

### 🏗️ Function: `load_graph_from_json` (Graph Reconstruction)

This function reconstructs a live **NetworkX** graph object from a serialized JSON file.

#### 🛠 Key Operations:
* **Encoding Resilience**: Uses `utf-8-sig` to ensure compatibility with various Windows-based text encodings.
* **Object Mapping**: Maps JSON 'nodes' to NetworkX nodes and JSON 'links' to weighted edges.
* **Metadata Retention**: Preserves the `weight` and `title` (relationship description) attributes, which are essential for the **Data Servant** agent to prioritize information.

#### 📍 Practical Use:
It transforms the static relationship data extracted from `lawsuit.txt` into a navigable mathematical structure, enabling the system to perform complex pathfinding and community detection.

In [6]:
# function to load df from json for chunk retrieval based on node and chunk id
def load_chunks_dataframe(json_file_path):
    try:
        with open(json_file_path, 'r', encoding='utf-8-sig') as file:
            data = json.load(file)

        # Extracting chunks and their IDs
        chunks = []
        for link in data['links']:
            chunk_ids = link.get('chunk_id', '').split(',')
            text = link.get('title', '')  # Assuming 'title' contains the text associated with the chunk
            for chunk_id in chunk_ids:
                if chunk_id:
                    chunks.append({'chunk_id': chunk_id, 'text': text})

        return pd.DataFrame(chunks)

    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in case of an error

# function to get top-n contextually closest nodes and edges for a given node
def get_top_contextual_nodes_edges(graph, node, top_n=5):
    if node not in graph:
        return []
    neighbors = [(n, graph[node][n]['weight']) for n in graph.neighbors(node)]
    sorted_neighbors = sorted(neighbors, key=itemgetter(1), reverse=True)
    return sorted_neighbors[:top_n]

### 🔍 Retrieval Logic: Connecting the Graph to the Evidence

We use these functions to navigate our legal knowledge base during the **Data Servant** phase.

#### 🛠 Logic Breakdown:
* **`load_chunks_dataframe`**: Acts as a bridge between structured graph data and unstructured text. It maps unique `chunk_id` identifiers to the original legal text, enabling the AI to cite its sources.
* **`get_top_contextual_nodes_edges`**: Performs a localized graph traversal. By identifying the most "weighted" neighbors of a legal entity, we provide the LLM with immediate context regarding related parties, claims, and evidence.

#### 📍 Practical Application:
If a user asks about **"Negligence"** in the *Smith v. Johnson* case, these functions allow the system to identify that "Negligence" is a claim by the Plaintiff (Graph Logic) and then retrieve the specific paragraph from `lawsuit.txt` where that claim is defined (Chunk Logic).

In [9]:
# Change these lines to use your absolute path
base_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output"
graph_json = os.path.join(base_path, "graph_data.json")

# Load the graph
graph = load_graph_from_json(graph_json)

# 🔥 SAFETY GATE: Stop here if the graph didn't load
if graph is None:
    raise FileNotFoundError(f"❌ Error: Could not find {graph_json}. Have you run the Graph Creation script yet?")
# load the chunk_dataframe
chunks_dataframe = load_chunks_dataframe(graph_json)

# ==============================
# Calculate tf-idf-scores for all nodes in context
# ==============================
#region tf-idf logic for nodes context

#Calculate Term Frequency (TF) for nodes context
# Initialize a dictionary to count occurrences
node_context_count = {}
total_contexts = set()

# Iterate over the edges in the NetworkX graph
for (source, target, data) in graph.edges(data=True):
    context = data.get('title', 'No title')
    total_contexts.add(context)

    if source not in node_context_count:
        node_context_count[source] = {}
    if target not in node_context_count:
        node_context_count[target] = {}

    node_context_count[source].setdefault(context, 0)
    node_context_count[target].setdefault(context, 0)
    node_context_count[source][context] += 1
    node_context_count[target][context] += 1

# Calculate Inverse Document Frequency (IDF)
idf_scores = {}
num_contexts = len(total_contexts)

for node, contexts in node_context_count.items():
    idf_scores[node] = math.log(num_contexts / len(contexts))

# Calculate TF-IDF Scores
tf_idf_scores = {}

for node, contexts in node_context_count.items():
    tf_idf_scores[node] = {}
    for context, count in contexts.items():
        tf = count / len(contexts)
        idf = idf_scores[node]
        tf_idf_scores[node][context] = tf * idf
# for custom algo:
# Extract a single TF-IDF score per node (e.g., the maximum score)
single_tf_idf_scores = {node: max(contexts.values()) for node, contexts in tf_idf_scores.items()}

# endregion

### 📊 Statistical Weighting: TF-IDF Node Analysis

This block implements a custom **TF-IDF (Term Frequency-Inverse Document Frequency)** logic applied to the graph topology. 

#### 🛠 Mathematical Goals:
* **Relevance Filtering**: We identify which legal entities are central to the specific "story" of the `lawsuit.txt` vs. which are general legal filler.
* **Contextual Scoring**: By analyzing the `title` of each edge as a document, we calculate how unique a node is within its neighborhood.
* **Priority Ranking**: The resulting `single_tf_idf_scores` are used by our retrieval algorithm to decide which nodes should be presented to the LLM as "Primary Evidence."

#### 📍 Practical Result:
When querying about damages, this logic ensures that the nodes representing **"$15,000"** and **"financial harm"** are ranked higher than generic nodes like **"Date"** or **"Case Number"**.

In [10]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
import pandas as pd

# 1. Configure the Global Settings (using your local BAAI model and Groq)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
# Replace 'your_groq_api_key' with your actual variable name
Settings.llm = Groq(model="llama-3.3-70b-versatile", api_key=groq_api_key)

# 2. Updated function to generate embeddings using HuggingFace
def generate_embeddings(df, column_name):
    """
    Generates embeddings for a specific column in a dataframe 
    using the locally hosted BAAI/bge-small-en-v1.5 model.
    """
    print(f"⏳ Generating embeddings for column: {column_name}...")
    
    # Access the embedding model from LlamaIndex Settings
    embed_model = Settings.embed_model
    
    # Get the list of texts from the dataframe
    texts = df[column_name].tolist()
    
    # Generate embeddings (HuggingFaceEmbedding uses get_text_embedding_batch)
    embeddings = embed_model.get_text_embedding_batch(texts)
    
    return embeddings

# 3. Apply the embedding generation to the graph dataframe
# This ensures node_1, node_2, and the relationship edges all share the same vector space
df_graph['node_1_embedding'] = generate_embeddings(df_graph, 'node_1')
df_graph['node_2_embedding'] = generate_embeddings(df_graph, 'node_2')
df_graph['edge_embedding'] = generate_embeddings(df_graph, 'edge')

print("🚀 Status: All graph embeddings generated successfully using BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Generating embeddings for column: node_1...
⏳ Generating embeddings for column: node_2...
⏳ Generating embeddings for column: edge...
🚀 Status: All graph embeddings generated successfully using BAAI/bge-small-en-v1.5


### 🔢 Feature Engineering: Vectorizing the Knowledge Graph

This block performs the critical task of transforming our symbolic Knowledge Graph into a **Latent Vector Space**.

#### 🛠 Core Mechanics:
* **Semantic Encoding**: By converting entities (Nodes) and relationships (Edges) into 1536-dimensional vectors, we enable mathematical similarity comparisons.
* **Triplet Vectorization**: We encode the `node_1`, `node_2`, and `edge` columns separately. This allows the retrieval engine to query the graph based on the "meaning" of the entities rather than exact string matching.
* **Cross-Model Alignment**: These embeddings serve as the bridge between our structured `graph.csv` and the unstructured vector search performed by our LLM agents.

#### 📍 Application:
This ensures that when a user asks about **"Legal Responsibilities,"** the system can mathematically navigate to nodes labeled **"Contractual Obligations"** or **"Duty of Care."**

In [12]:
# 1. Convert embeddings to a list for clustering
nodes, embeddings = zip(*node_embeddings_avg.items())
embeddings_list = list(embeddings)

# 2. 🔥 THE FIX: Calculate n_clusters dynamically
# We want 5 clusters, BUT if we have fewer than 5 nodes, 
# we use the total number of nodes instead.
num_nodes = len(nodes)
n_clusters = min(5, num_nodes) 

print(f"📊 Clustering {num_nodes} nodes into {n_clusters} groups...")

if n_clusters > 0:
    # 3. Perform K-Means clustering
    kmeans = KMeans(n_clusters=n_clusters, n_init=10)
    clusters = kmeans.fit_predict(embeddings_list)

    # 4. Map nodes to their clusters
    node_cluster_mapping = dict(zip(nodes, clusters))
    print("✅ Successfully mapped nodes to thematic clusters.")
else:
    print("⚠️ Warning: No nodes found to cluster.")
    node_cluster_mapping = {}

📊 Clustering 2 nodes into 2 groups...
✅ Successfully mapped nodes to thematic clusters.


### 🏷️ Unsupervised Learning: Semantic Topic Clustering

We implement **K-Means Clustering** to categorize legal entities into thematic groups without manual labeling.

#### 🛠 Technical Workflow:
* **Vector Aggregation**: We collect all embeddings for a specific node and calculate the **centroid (mean)** to represent its core semantic value.
* **Dimensional Clustering**: Using the `scikit-learn` KMeans implementation, we partition the 384-dimensional vector space (from our BAAI model) into 5 distinct clusters.
* **Structural Insight**: This allows the system to identify "communities" within the lawsuit, such as grouping all technical terms related to "Web Development" separate from "Legal Damages."

#### 📍 Application:
This clustering directly drives the **visual layout** of our GraphRAG, allowing users to navigate the `lawsuit.txt` by thematic regions rather than just a list of names.

### 🏷️ Robust Semantic Clustering

We have updated the clustering logic to handle varying dataset sizes dynamically.

#### 🛠 Logic Update:
* **Dynamic `n_clusters`**: We now use `min(5, num_nodes)`. This prevents the `ValueError` encountered when the graph contains fewer entities than the requested cluster count.
* **Initialization Safety**: Added an `if n_clusters > 0` check to ensure the script doesn't attempt to process an empty graph.
* **Parameter Tuning**: Set `n_init=10` explicitly to comply with updated `scikit-learn` defaults, ensuring stable cluster centroids.

#### 📍 Practical Result:
Whether analyzing a single-page case summary or an entire legal database, the system will now partition the knowledge graph into the most mathematically appropriate number of thematic groups.

In [13]:
def find_similar_nodes_and_edges(query, df, top_n=2):
    print(f"🔍 Searching graph for: {query}")
    
    # Use your local BAAI model instead of OpenAI
    query_embedding = Settings.embed_model.get_query_embedding(query)
    
    # Calculate similarity against the triplet (Node1, Node2, Edge)
    df['similarity'] = df.apply(lambda row: cosine_similarity(
        [row['node_1_embedding']] + [row['node_2_embedding']] + [row['edge_embedding']],
        [query_embedding])[0][0], axis=1)

    # Return the best matches
    similar_nodes_df = df.nlargest(top_n, 'similarity')
    return similar_nodes_df[['node_1', 'node_2', 'edge', 'chunk_id', 'similarity']]

### 🎯 Semantic Graph Retrieval

This function implements the core "Search" logic of our GraphRAG system.

#### 🛠 Technical Mechanics:
* **Vector Comparison**: It utilizes **Cosine Similarity** to compare user queries against the three-part embeddings of our legal triplets.
* **Fuzzy Matching**: By searching in vector space, the system can identify relevant legal concepts even when the user's terminology differs from the source text.
* **Evidence Mapping**: Returns the `chunk_id` alongside the graph nodes, allowing the **Senior Law Consultant** to verify graph-based insights against the raw document evidence.

#### 📍 Practical Example:
If the user asks about **"Financial Penalties,"** this function identifies that the relationship `[Michael Johnson] -> [is ordered to pay] -> [$15,000]` is the most mathematically similar context.

In [14]:
# function to get text_chunks from chunk Id based on similar nodes with cosine similarity
def get_text_chunks(chunk_ids, chunks_dataframe):
    # Retrieve rows where chunk_id is in the given set of chunk_ids
    relevant_rows = chunks_dataframe[chunks_dataframe['chunk_id'].isin(chunk_ids)]

    # Filter out chunks containing specific strings
    filtered_chunks = relevant_rows[~relevant_rows['text'].str.contains("chunk contextual proximity|global contextual proximity", regex=True)]

    return filtered_chunks['text'].tolist()

### 📖 Function: `get_text_chunks` (Evidentiary Retrieval)

This function completes the RAG cycle by retrieving raw textual evidence based on structural graph matches.

#### 🛠 Logic Flow:
* **Relational Mapping**: Uses the unique `chunk_id` found in the Knowledge Graph to index into our processed text dataframe.
* **Data Sanitization**: Implements a regex filter to remove "Graph-only" metadata (like proximity markers), ensuring the LLM only receives clean, human-readable legal text.
* **Context Synthesis**: Converts the filtered dataframe rows into a Python list, which serves as the primary "Source Text" for our final legal response.

#### 📍 Practical Application:
When the system identifies a "Guilty" verdict in the graph, this function retrieves the specific **Judge's Remarks** from the original text to provide the "Why" behind the verdict.

In [15]:
# function to extract cosine scores for vector searched nodes based on query
def extract_cosine_scores(similar_nodes_df):
    cosine_scores = {}
    for _, row in similar_nodes_df.iterrows():
        nodes = [row['node_1'], row['node_2']]
        for node in nodes:
            cosine_scores[node] = max(row['similarity'], cosine_scores.get(node, 0))
    return cosine_scores

### 🎯 Function: `extract_cosine_scores` (Relevance Mapping)

This function extracts and normalizes the semantic similarity scores for individual legal entities.

#### 🛠 Key Mechanics:
* **Entity Isolation**: It breaks down relational triplets into individual nodes, ensuring both the subject and object are assigned a relevance value.
* **Peak Score Retention**: By using the `max()` function, it ensures that an entity's relevance is determined by its strongest semantic link to the user's query.
* **Data Formatting**: Converts complex dataframe results into a lightweight Python dictionary for rapid lookup during the context generation phase.

#### 📍 Practical Application:
This ensures that if **"John Smith"** is mentioned in several relevant parts of the lawsuit, his importance in the final answer is scaled to the most significant piece of evidence found.